# 98 — Campaign B status dashboard（**読み取り専用**）

M2 / キュー / M3–M6 / backlog / stale lease を表示する。

- **計算状態・lock・lease・queue は変更しない。**
- **`validate_persistent_root()` は呼ばない**（書き込み probe が必要で、disk quota / Errno 122 で落ちるため）。
- persist root が存在するディレクトリかを確認するだけ。欠けていれば警告して続行。
- disk quota で書き込みがブロックされていても、既存成果物の読み取り表示は可能。


In [ ]:
from pathlib import Path
import os, sys, json

BUNDLE_NAME = 'validated_4d_su2_rg_codex_bundle'
explicit = os.environ.get('VALIDATED_RG_PROJECT_ROOT')
candidates = ([Path(explicit)] if explicit else []) + [
    Path.cwd(), Path.cwd().parent, Path.cwd() / BUNDLE_NAME,
    Path('/notebooks') / BUNDLE_NAME, Path('/notebooks'),
]
PROJECT_ROOT = next((
    p.expanduser().resolve()
    for p in candidates
    if (p.expanduser() / 'src' / 'campaign_b' / 'pipeline_status.py').is_file()
), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Pull latest main; src/campaign_b/pipeline_status.py not found.')
PERSIST_ROOT = Path(
    os.environ.get('VALIDATED_RG_PERSIST_ROOT', '/storage/validated_4d_su2_rg')
).expanduser().resolve()
os.environ['VALIDATED_RG_PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['VALIDATED_RG_PERSIST_ROOT'] = str(PERSIST_ROOT)
# Harmless for other imports; 98 itself never calls validate_persistent_root().
os.environ.setdefault('VALIDATED_RG_PERSIST_ACK', 'I_CONFIRM_THIS_PATH_IS_PERSISTENT')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT', PROJECT_ROOT)
print('PERSIST_ROOT', PERSIST_ROOT)


In [ ]:
from src.campaign_b.pipeline_status import collect_pipeline_status, format_status_text

# Read-only: do NOT call validate_persistent_root() (it writes a probe and can fail
# with PersistentRootError / Errno 122 under disk quota).
if not PERSIST_ROOT.exists():
    print(f'WARNING: persist root does not exist: {PERSIST_ROOT}')
elif not PERSIST_ROOT.is_dir():
    print(f'WARNING: persist root is not a directory: {PERSIST_ROOT}')
else:
    print(f'persist root OK (read-only check): {PERSIST_ROOT}')

STATUS = collect_pipeline_status(PERSIST_ROOT)  # write_status_snapshot=False by default
print(format_status_text(STATUS))


In [ ]:
# Compact view
print(json.dumps({
    'scanned_at': STATUS.get('scanned_at'),
    'm2': STATUS.get('m2'),
    'queues': STATUS.get('queues'),
    'candidate_states': STATUS.get('candidate_states'),
    'leases': STATUS.get('leases'),
    'selected_packages': (STATUS.get('selected') or {}).get('selected_packages'),
}, indent=2, ensure_ascii=False, default=str))
